In [1]:
import numpy as np
import pandas as pd

# CUSTOMER CHURN PREDICTION MODEL

df = pd.read_csv('Churn_Modeling.csv')

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   RowNumber        10000 non-null  int64  
 1   CustomerId       10000 non-null  int64  
 2   Surname          10000 non-null  object 
 3   CreditScore      10000 non-null  int64  
 4   Geography        10000 non-null  object 
 5   Gender           10000 non-null  object 
 6   Age              10000 non-null  int64  
 7   Tenure           10000 non-null  int64  
 8   Balance          10000 non-null  float64
 9   NumOfProducts    10000 non-null  int64  
 10  HasCrCard        10000 non-null  int64  
 11  IsActiveMember   10000 non-null  int64  
 12  EstimatedSalary  10000 non-null  float64
 13  Exited           10000 non-null  int64  
dtypes: float64(2), int64(9), object(3)
memory usage: 1.1+ MB


In [2]:
# Drop unnecessary features

df= df.drop(['CustomerId', "Surname", "RowNumber"], axis=1)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   CreditScore      10000 non-null  int64  
 1   Geography        10000 non-null  object 
 2   Gender           10000 non-null  object 
 3   Age              10000 non-null  int64  
 4   Tenure           10000 non-null  int64  
 5   Balance          10000 non-null  float64
 6   NumOfProducts    10000 non-null  int64  
 7   HasCrCard        10000 non-null  int64  
 8   IsActiveMember   10000 non-null  int64  
 9   EstimatedSalary  10000 non-null  float64
 10  Exited           10000 non-null  int64  
dtypes: float64(2), int64(7), object(2)
memory usage: 859.5+ KB


In [3]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Split features

X = df.drop('Exited', axis=1)
y = df['Exited']

# Encode categorical features
X = pd.get_dummies(X, columns=['Geography',"Gender"], drop_first=True,)

# Split into training and testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


# Scaling data
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [4]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout

# Building the ANN model
model = Sequential([
    Dense(64, activation='relu', input_shape=(X_train_scaled.shape[1],)),
    Dropout(0.3),

    Dense(32, activation='relu'),
    Dropout(0.3),

    Dense(16, activation='relu'),

    Dense(1, activation='sigmoid')

])

# Compliling the model
model.compile(
    optimizer ='adam',
    loss = 'binary_crossentropy',
    metrics = ['accuracy']
)

# Ths shows us the model structure
model.summary()

/opt/anaconda3/envs/ml/lib/python3.11/site-packages/keras/src/layers/core/dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │           768 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,393 (13.25 KB)

 Trainable params: 3,393 (13.25 KB)

 Non-trainable params: 0 (0.00 B)

In [5]:
from tensorflow.keras.callbacks import EarlyStopping

# Here we add EarlyStopping Callback to prevent overfitting
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

# Fit model
history = model.fit(X_train_scaled,y_train, validation_split=0.2, epochs=50, batch_size=32,callbacks=[early_stop], verbose=1)

# Evaluate model
loss, accuracy = model.evaluate(X_test_scaled,y_test)

print(f"Test Loss: {loss}")
print(f"Test Accuracy: {accuracy}")

Epoch 1/50
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.7875 - loss: 0.4971 - val_accuracy: 0.8138 - val_loss: 0.4278
Epoch 2/50
200/200 ━━━━━━━━━━━━━━━━━━━━ 0s 817us/step - accuracy: 0.8037 - loss: 0.4538 - val_accuracy: 0.8206 - val_loss: 0.4137
Epoch 3/50
200/200 ━━━━━━━━━━━━━━━━━━━━ 0s 907us/step - accuracy: 0.8095 - loss: 0.4360 - val_accuracy: 0.8325 - val_loss: 0.4022
Epoch 4/50
200/200 ━━━━━━━━━━━━━━━━━━━━ 0s 735us/step - accuracy: 0.8197 - loss: 0.4225 - val_accuracy: 0.8444 - val_loss: 0.3865
Epoch 5/50
200/200 ━━━━━━━━━━━━━━━━━━━━ 0s 764us/step - accuracy: 0.8308 - loss: 0.4020 - val_accuracy: 0.8444 - val_loss: 0.3760
Epoch 6/50
200/200 ━━━━━━━━━━━━━━━━━━━━ 0s 731us/step - accuracy: 0.8306 - loss: 0.3943 - val_accuracy: 0.8481 - val_loss: 0.3642
Epoch 7/50
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.8389 - loss: 0.3802 - val_accuracy: 0.8562 - val_loss: 0.3568
Epoch 8/50
200/200 ━━━━━━━━━━━━━━━━━━━━ 0s 868us/step - accuracy: 0.8439 - loss: 0.3734 - val_

In [6]:
from sklearn.metrics import confusion_matrix

# Since this outputs an array of probs we need the next line
y_pred_prob = model.predict(X_test_scaled)

# Converts probs to either true or false and then astype(int) converts them to 0 or 1
y_pred = (y_pred_prob > 0.5).astype(int)

cm = confusion_matrix(y_test,y_pred)

print(cm)

63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 699us/step
[[1556   51]
 [ 228  165]]
